   `schedule(void)` is the central coordinator of the context switch you need
   to describe ... dictating exactly how 

Part 5: Explain how thread scheduling in PintOS currently works in roughly 300 words. Include the chain of execution of function calls.
(Hint: we expect you to at least mention which functions participate in a context switch, how they interact, how and when the thread state is modified and the role of interrupts.)

---

   The prompt is targeting the mechanical handover of the CPU--the context 
   switch itself--rather than the entire biographical lifespan of a thread. 
   While thread creation and death are part of the broader ssytem, the phrase
   "how thread scheduling... works" combined with the hints means they want you
   to detail the exact sequence of events that causes one running thread to step
   down and a waiting thread to take over. This sequence is identical whether
   the switch is involuntary (a timer tick running out) or voluntary (a thread
   blocking for a lock or exiting)...

   To hit the ubric's requirements, you should start by explaining the role of 
   interrupts. In PintOS, preemption is driven by the hardware timer, which 
   periodically fires an external interrupt. This invokes `thread_tick()`, which
   tracks the running thread's CPU time and, if its slice has expired, calls 
   `intr_yield_on_return()` to trigger a `thread_yield()` as the interrupt 
   concludes. Crucially, before the actual scheduling can begin, the system
    must disable interrupts to prevent fatal race conditions and manually change
   outgoing thread's state from `THREAD_RUNNING` to `THREAD_READY` (or 
   `THREAD_BLOCKED` / `THREAD_DYING` for voluntary switches)

   Next, you must outline the specific chain of function calls that executes the
   swap. Once the thread's state is updated and interrupts are disabled, 
   functions like `thread_yield()` call into `schedule()`. This function selects
   the next thread to run from the ready queue and passes both the outgoing and
   incoming threads into `switch_threads()`. This assembly routine performs the
   physical context switch: it pushes the current CPU registers onto the 
   outgoing thread's stack, swaps the CPU's stack pointer (`%esp`) to the 
   incoming thread's stack, and pops the incoming thread's previously saved 
   registers back into the CPU. 

   Finally, the explanation must cover how the state is finalized after the 
   assembly routine finishes. Because the stack pointer was swapped, the CPU
   returns from `switch_threads()` and effectively "wakes up" inside the new
   thread's execution context. The new thread immediately executes 
   `thread_schedule_tail()` which completes the process by officially changing 
   the new thread's state to `THREAD_RUNNING`. This tail function also handles
   a critical cleanup step: if the thread that just yielded the CPU was in the 
   `THREAD_DYING` state, its memory page is permanently freed here, as it could 
   not be destroyed while it was actively executing the switch ... 



---

-- THREAD SCHEDULING in Pintos is the kernel mechanism that decides which 
   runnable thread gets to execute on the CPU next. 

-- Pintos thead scheduling involves managing the execution, state transitions, 
   and CPU allocation for kernel threads using a ready list, context switching,
   and priority or multi-level feedback mechanism...

   CORE COMPONENTS
   - THREAD STATES: Each thread `struct thread` tracks its current state as 
     `running`, `ready`, `blocked`, or `dying`
   - THE READY LIST: A list where all threads in the `ready` state wait to be
     scheduled in the CPU.
   - THE SCHEDULER (`schedule()`): Picks the next thread to run. If no runnable
     threads exist, it runs a special `idle` thread
   - CONTEXT SWITCHING: Saves the current thread's CPU registers, switching the
     stack pointer, and restores the next thread's execution context via 
     assembly routines.   


---

A VOLUNTARY SWITCH: THE THREAD WAITS FOR AN EVENT
   `sema_down` 
   - It turns interrupts off. While the sempahore's value is 0, it adds the
     current thread to `sema->waiters` through `elem`, then calls 
     `thread_block()`.

   ``

   ... ``

   ... the crucial detail is that `intr_disable()` inside `sema_down()` is 
   extremely temporary. The interrupts are only turned off for the few 
   microsecond it takes to safely chek `sema->value` and modify the `waiters` 
   list without interferece .... it calls `intr_set_level(old_level)`, which
   immediately restores the CPU's interrupts back to their previous active
   state before handing contrl back to your code. Therefore, when a thread
   successfully acquires the lock and enters its critical section, interrupts 
   are fully enabled again. If a thread gets stuck in an accdiental ... 

   `intr_yield_on_return()` cannot switch threads itself, because code inside an
   interrupt handler must not switch threads or sleep. It only asserts that it runs
   in external interrupt conext, and sets the flag `yield_on_return`. That flag
   is ready by `intr_handler`, the function that started this whole chain: when 
   the timer fires, the CPU jumps to an interrupt stub in `intr-stubs.S`, which
   calls `intr_handler`, which looks up the registered handler by the interrupt's
   vector number and calls it, here `timer_interrupt`. Once that handler returns,
   `intr_handler` leaves interrupt context, acknowledges the interrupt, and, 
   since `yield_on_return..`

---

   ... advising you to use `PRIu64` because `uint64_t` is guaranteed to be a 
   64-bit unsigned integer, but `unsignhed long long` is not the only type that 
   can represent a 64-bit integer across different computer architecture. 

   Using `%llu` might work on ... can cause compilation warnings, errors, or 
   broken output when your code is compiled on a different system. 

WHY `%llu` CAN FAIL
   The C99 standard guarantees that a `uint64_t` is exactly 64 bits, but it 
   allows different os AND cpu architectures to choose which underlying standard
   type matches that size. 

   - ON A 64-BIT LINUX SYSTEM: A `uint64_t` is typicaly defined as an
     `unsigned long int`

---

   In context of testing code, REPRODUCIBILITY means that a test will always 
   yield the exact same result (pass or fail) when run under the exact same
   conditions. 

   If you execute a test against a specific version of code, using the same 
   initial conitions, in the same environment, it must produce identical 
   behavior every single time. When a test ... or passes locally but fails in 
   CI/CD pipelines, you are dealing with an ... Tests that lack reproducibility
   are referred to as FLAKY TESTS...
   

---

   ... A commit message should describe THE CHANGE IN THAT COMMIT, so that 
   someone who reads `git log` later knows what each commit did ...
   and it becomes wrong as soon as the next commit lands

   ... Use the convention the Git slides recommend, Conventional Commits, because
   the markers grade your commit messages: 


   - `feat`: New behaviour
   - `fix`: A bug fix
   - `refactor`: A code change that keeps the behaviour
   - `test`: Tests only
   - `docs`: Documentation, for example `devices.md`
   - `style`: Formatting only
   - `chore`: Tooling and housekeeping, like `.gitignore` change